In [0]:
from google.colab import drive
drive.mount('/content/drive')

In [0]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import time, datetime
import re
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV

**1.DATA GATHERING**
    **(Loading Files into dataframes)**

In [0]:
a = pd.read_csv('/content/drive/MyDrive/Dataset/business.csv')
a.head(5)

In [0]:
b = pd.read_csv('/content/drive/MyDrive/Dataset/economy.csv')
b.head(5)


**2.PREPROCESSING**

In [0]:
a["class"] = "business"
b["class"] = "economy"

In [0]:
new = b.append(a,ignore_index = True)
new.head(6)

In [0]:
new.num_code = new.num_code.astype("str")
new["flight"] = new["ch_code"] +"-"+ new["num_code"]
new.drop(["ch_code","num_code"],axis = 1,inplace = True)

In [0]:
new.rename({"dep_time": "departure_time", "from": "source_city", 
            "time_taken": "duration", "stop": "stops", "arr_time": "arrival_time",
           "to":"destination_city"}, axis = 1, inplace = True)

In [0]:
dd = pd.DataFrame(new["date"].str.split("-",expand = True).to_numpy().astype(int),columns = ["day","month","year"])
new["days_left"] = np.where(dd["month"] > 2, dd["day"] +18, np.where(dd["month"] == 2, dd["day"] -10, dd["day"]))
new.head(6)

In [0]:
new.drop("date",axis = 1,inplace = True)

In [0]:
s = (pd.to_datetime(new["departure_time"]).dt.hour % 24 + 4) // 4 #give numbers from 1 to 6 #(return a series)
s.replace({1: 'Late Night', 2: 'Early Morning', 3: 'Morning', 
                      4: 'Afternoon', 5: 'Evening', 6: 'Night'} ,inplace = True) # to replace values 1:latenight to 6: night
new["departure_time"] = s 
new.head(5)

In [0]:
temp = pd.DataFrame(new["arrival_time"].str.split(":",expand = True).to_numpy().astype(int), 
                    columns = ["hour","minute"])
new["arrival_time"] = pd.cut(x = temp["hour"], bins = 6, labels = 
                             ["Late Night","Early Morning","Morning", "Afternoon", "Evening", "Night"])
new.head(5)

In [0]:
temp = pd.DataFrame(new["duration"].str.split(expand = True).to_numpy().astype(str), 
                    columns = ["hour","minute"])
temp["hour"] = temp["hour"].apply(lambda x: re.sub("[^0-9]","",x)).astype(int)
temp["minute"] = temp["minute"].apply(lambda r: re.sub("[^0-9]","",r))
temp["minute"] = np.where(temp["minute"] == "", 0, temp["minute"])
temp["minute"] = temp["minute"].astype(int) #converting data type
new["duration"] = np.around((temp["hour"] + (temp["minute"]/60)),2)
new.head(5)

In [0]:
new["stops"] = new["stops"].apply(lambda r: re.sub("[^0-9]","",r))
new["stops"] = np.where(new["stops"] == "", 0, new["stops"])
new["stops"] = new["stops"].astype(int)
new.head(10)

In [0]:
new["price"] = new["price"].apply(lambda r: re.sub("[^0-9]","",r))
new["price"] = new["price"].astype(int)
new.head(6)

**3.EXPLORATORY DATA ANALYSIS**

***3.1 HANDLING MISSING VALUES***




In [0]:
df = new
df.sample(10)

In [0]:
df.isna().sum()

In [0]:
df.info()

In [0]:
df.isna().any()

In [0]:
df.isnull().sum()

In [0]:
df.dtypes

In [0]:
df.describe()

In [0]:
df.duplicated().sum()

***3.2 DATA VISUALIZATION***

***3.2 (A) UNIVARIATE ANALYSIS***

In [0]:
Uniq_Airline = df.airline.unique()
Uniq_Airline

In [0]:
df["airline"].value_counts()

In [0]:
plt.figure(figsize = (10,5))
df["airline"].value_counts().plot(kind = "bar",cmap = "flag")
plt.title('Airlines',fontsize=15)
plt.show()

In [0]:
plt.figure(figsize = (15,5))
df['source_city'].value_counts().plot(kind = "pie", textprops={'color':'black'}, autopct = "%.2f",cmap = "summer" )
plt.title('Source City',fontsize=15)
plt.show()

In [0]:
plt.figure(figsize = (15,5))
df['destination_city'].value_counts().plot(kind = "pie",autopct = "%.2f",cmap = "cool" )
plt.title('Destination City',fontsize=15)
plt.show()

In [0]:
plt.figure(figsize = (15,5))
df['arrival_time'].value_counts().plot(kind = "pie",autopct = "%.2f",cmap = "rainbow" )
plt.title('Arrival Time',fontsize=15)
plt.show()

In [0]:
plt.figure(figsize = (15,5))
df["departure_time"].value_counts().plot(kind = "pie",autopct = "%.2f",cmap = 'cool')
plt.title("Departure_Time")
plt.show()

In [0]:
plt.figure(figsize = (5,5))
df['stops'].value_counts().plot(kind = "pie", textprops={'color':'black'}, autopct = "%.2f",cmap = "spring" )
plt.title('Flight Stops',fontsize=15)
plt.legend([1,0,2])
plt.show()

In [0]:
plt.figure(figsize = (5,5))
df['class'].value_counts().plot(kind = "pie",autopct = "%.2f",cmap = "summer" )
plt.title('Ticket Class',fontsize=15)
plt.show()

***3.2 (B) BIVARIATE ANALYSIS***

In [0]:
plt.figure(figsize = (15,5))
sns.barplot(data = df, x = "airline" , y = "price")

In [0]:
plt.figure(figsize = (15,5))
sns.barplot(x= "days_left",y = "price",data = df)

In [0]:
fig,ax=plt.subplots(1,2,figsize=(18,6))
sns.barplot(x='departure_time',y='price',data=df,ax=ax[0])
sns.barplot(x='arrival_time',y='price',data=df,ax=ax[1])

In [0]:
fig,ax=plt.subplots(1,2,figsize=(18,6))
sns.barplot(x='source_city',y='price',data=df,ax=ax[0])
sns.barplot(x='destination_city',y='price',data=df,ax=ax[1])

In [0]:
plt.figure(figsize = (10,9))
sns.barplot(data = df, x = "source_city" , y = "price" , hue = "destination_city")

In [0]:
plt.figure(figsize = (10,5))
sns.barplot(data = df, x = "class" , y = "price")

In [0]:
plt.figure(figsize = (10,5))
sns.barplot(data = df, x = "airline" , y = "price" , hue = "class")

**4.FEATURE ENGINEERING**

In [0]:
df.drop_duplicates(inplace = True)

In [0]:
df1 = df.drop("flight", axis = 1)
df1.head(5)

In [0]:
df1["departure_time"].replace({'Late Night':0,'Early Morning':1,'Morning':2,
                               'Afternoon':3,'Evening':4,'Night':5},inplace=True)
df1["arrival_time"].replace({'Late Night':0,'Early Morning':1,'Morning':2,
                             'Afternoon':3,'Evening':4,'Night':5},inplace=True)
df1["class"].replace({"economy":0,"business":1},inplace=True)
df1.head(6)

In [0]:
ohe = OneHotEncoder()
df1[list(df1["airline"].unique())] = ohe.fit_transform(df1[["airline"]]).A 
df1 = pd.concat([df1,pd.get_dummies(df1["destination_city"],prefix = "destination_city")], axis = 1)
df1 = pd.concat([df1,pd.get_dummies(df1["source_city"],prefix = "source_city")], axis = 1)
df1.drop(["airline","source_city","destination_city"],axis = 1,inplace=True)
df1.head(5)

**5.MODEL SELECTION AND MODEL TRAINING**

In [0]:
x = df1.drop("price", axis = 1)
y = df1["price"]

In [0]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.33,random_state=45)

In [0]:
scal = StandardScaler()
arr = scal.fit_transform(x_train)
x_train1 = pd.DataFrame(arr, columns = x_train.columns)

In [0]:
x_test1 = scal.transform(x_test)

In [0]:
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

In [0]:
def get_accuracy(model_name):
    model         = model_name
    model.fit(x_train1,y_train)
    y_pred_train  = model.predict(x_train1)
    mse_train     = mean_squared_error(y_train,y_pred_train)
    mae_train     = mean_absolute_error(y_train,y_pred_train)
    r2score_train = r2_score(y_train,y_pred_train)
    
    y_pred_test   = model.predict(x_test1)
    mse_test      = mean_squared_error(y_test,y_pred_test)
    mae_test      = mean_absolute_error(y_test,y_pred_test)
    r2score_test = r2_score(y_test,y_pred_test)
    
    return print(f"mse_train {mse_train} \nmae_train {mae_train} \nr2score_train {r2score_train} \nmse_test {mse_test} \nmae_test {mae_test} \nr2score_test {r2score_test}")

In [0]:
for model_name,model in [("LinearRegression",LinearRegression()),("DecisionTreeRegressor",DecisionTreeRegressor()),("RandomForestRegressor",RandomForestRegressor()),("KNeighborsRegressor",KNeighborsRegressor())]:
    print(model_name)
    print(get_accuracy(model))
    print("*"*100)

**6.MODEL EVALUATION**

In [0]:
random_model = RandomForestRegressor()
random_model.fit(x_train1,y_train)

***6.1 ACCURACY IN TRAINING TIME***

In [0]:
y_pred_train  = random_model.predict(x_train1)
mse_train     = mean_squared_error(y_train,y_pred_train)
mae_train     = mean_absolute_error(y_train,y_pred_train)
r2score_train = r2_score(y_train,y_pred_train)
print(r2score_train)

***6.2 ACCURACY IN TESTING TIME***

In [0]:
y_pred_test  = random_model.predict(x_test1)
mse_test     = mean_squared_error(y_test,y_pred_test)
mae_test     = mean_absolute_error(y_test,y_pred_test)
r2score_test = r2_score(y_test,y_pred_test)
print(r2score_test)